# Stage 2: Targeted LoRA Finetuning on Hallucination Heads

**Goal:** Surgically fine-tune only the 32 identified hallucination heads using contrastive COCO caption pairs.

**Key design decisions vs Stage 1:**
- Uses **4-bit quantization (QLoRA)** — safe here because Stage 2 does *not* extract attention weights
- LoRA applied only to Q/K/V of the **17 layers** that contain at least one hallucination head
- **Gradient masking** restricts updates to only the 32 head-slices within those matrices
- **DPO loss**: GT captions = chosen, model's own hallucinated captions = rejected
- Reference logprobs computed on-the-fly via `disable_adapter_layers()` (no separate copy of the model)

**Prerequisites:** Run Stage 1 first. This notebook reads from `llava_hallucination_heads/` on Drive:
- `results/final_hallucination_heads.json`
- `cache/screening_state.pkl`
- `cache/selected_imgs.json`
- `coco/val2014_subset/` (images) and `coco/annotations/` (captions + instances)

## 0. Install dependencies
**Run once, then Runtime → Restart session. Skip on subsequent runs.**

In [1]:
# # RUN ONCE then Runtime -> Restart session
# !nvidia-smi --query-gpu=name,memory.total --format=csv
# !pip install -q "transformers>=4.47" "accelerate>=0.33" "tokenizers>=0.21"
# !pip install -q peft bitsandbytes
# !pip install -q "numpy<2.0" pandas pillow tqdm pycocotools spacy sentencepiece
# !python -m spacy download en_core_web_sm -q
# print('Done. Runtime -> Restart session, then skip this cell.')

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9

## 1. Imports and Drive setup
**Start here after session restart.**

In [1]:
import torch
import numpy as np
import pandas as pd
import json, os, gc, pickle, random, re
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/llava_hallucination_heads'
COCO_DIR = f'{WORK_DIR}/coco'
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
os.makedirs(f'{WORK_DIR}/cache',   exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Load LLaVA-1.5-7B in 4-bit (QLoRA)

Stage 1 used fp16 *without* 4-bit because quantization breaks attention weight extraction.
Stage 2 only needs **logits** for the DPO loss, so 4-bit is safe and saves ~10 GB VRAM.

In [2]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    device_map={'': 0},
)

# prepare_model_for_kbit_training must run BEFORE get_peft_model:
# - enables gradient checkpointing
# - casts layer norms to fp32 for numerical stability
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded. VRAM used: 4.59 GB


## 3. Resolve model constants and decoder layers

In [3]:
# Navigate model internals — handles both old and new transformers layouts
if hasattr(model, 'language_model'):
    lm = model.language_model
elif hasattr(model, 'model') and hasattr(model.model, 'language_model'):
    lm = model.model.language_model
else:
    raise RuntimeError('Cannot find language_model')

if hasattr(lm, 'model') and hasattr(lm.model, 'layers'):
    decoder_layers = lm.model.layers
elif hasattr(lm, 'layers'):
    decoder_layers = lm.layers
else:
    raise RuntimeError('Cannot find decoder layers')

text_cfg         = model.config.text_config
NUM_LAYERS       = text_cfg.num_hidden_layers
NUM_HEADS        = text_cfg.num_attention_heads
HEAD_DIM         = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID   = model.config.image_token_index
vision_cfg       = model.config.vision_config
NUM_IMAGE_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2  # 576
PROMPT_TEMPLATE  = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

print(f'Layers: {NUM_LAYERS}, Heads/layer: {NUM_HEADS}, head_dim: {HEAD_DIM}')
print(f'Image token id: {IMAGE_TOKEN_ID}, Image tokens: {NUM_IMAGE_TOKENS}')
print(f'Decoder layers: {len(decoder_layers)}')
assert len(decoder_layers) == NUM_LAYERS

Layers: 32, Heads/layer: 32, head_dim: 128
Image token id: 32000, Image tokens: 576
Decoder layers: 32


## 4. Load Stage 1 artifacts

In [4]:
from pycocotools.coco import COCO

# --- Hallucination heads from Stage 1 ---
with open(f'{WORK_DIR}/results/final_hallucination_heads.json') as f:
    final_list = json.load(f)

hal_heads_by_layer = defaultdict(set)
for h in final_list:
    hal_heads_by_layer[h['layer']].add(h['head'])
target_layers = sorted(hal_heads_by_layer.keys())

print(f'Hallucination heads: {len(final_list)} across {len(target_layers)} layers')
print(f'Target layers: {target_layers}')

# --- Per-image cache: model captions + content word labels from Stage 1 ---
CACHE_FILE = f'{WORK_DIR}/cache/screening_state.pkl'
with open(CACHE_FILE, 'rb') as f:
    stage1_state = pickle.load(f)
per_image_cache = stage1_state['per_image_cache']
print(f'Per-image cache: {len(per_image_cache)} entries')

# --- Selected images + ground-truth objects from Stage 1 ---
with open(f'{WORK_DIR}/cache/selected_imgs.json') as f:
    img_meta = json.load(f)
selected_imgs     = img_meta['ids']
img_to_gt_objects = {int(k): set(v) for k, v in img_meta['gt_objects'].items()}

# Rebuild img_id_to_path from COCO annotations
ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
coco = COCO(ann_path)
img_dir = f'{COCO_DIR}/val2014_subset'
img_meta_coco = coco.loadImgs(selected_imgs)
img_id_to_path = {m['id']: f"{img_dir}/{m['file_name']}" for m in img_meta_coco}

# COCO category names
cat_id_to_name = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

n_exist = sum(1 for p in img_id_to_path.values() if os.path.exists(p))
print(f'Images found: {n_exist}/{len(img_id_to_path)}')

Hallucination heads: 32 across 19 layers
Target layers: [7, 9, 12, 13, 14, 15, 16, 17, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31]
Per-image cache: 200 entries
loading annotations into memory...
Done (t=7.77s)
creating index...
index created!
Images found: 200/200


## 5. Build contrastive training pairs

- **Chosen (positive):** COCO ground-truth captions (up to 5 per image)
- **Rejected (negative):** Model's own hallucinated caption from Stage 1
- Only use images where Stage 1 identified at least one hallucinated content word
- **Train/eval split:** last 40 images held out for evaluation

In [5]:
# Load COCO GT captions (already in annotations zip from Stage 1)
cap_ann_path = f'{COCO_DIR}/annotations/captions_val2014.json'
coco_caps = COCO(cap_ann_path)
print(f'COCO captions loaded: {len(coco_caps.anns)} annotations')

# Fixed train/eval split — last 40 images for eval, first 160 for train
random.seed(42)
eval_img_ids  = set(selected_imgs[-40:])
train_img_ids = set(selected_imgs[:-40])

eval_images     = [img_id for img_id in selected_imgs[-40:] if img_id in img_id_to_path]
eval_gt_objects = [img_to_gt_objects[img_id] for img_id in eval_images]

# Build DPO pairs from train images
dpo_pairs = []
for img_id_raw, cache_info in per_image_cache.items():
    img_id = int(img_id_raw) if isinstance(img_id_raw, str) else img_id_raw
    if img_id not in train_img_ids:
        continue
    img_path = img_id_to_path.get(img_id)
    if not img_path or not os.path.exists(img_path):
        continue

    # Only use images where Stage 1 found hallucinations (rejected caption is meaningful)
    has_hallucination = any(
        cw['is_hallucinated'] for cw in cache_info.get('content_words', [])
    )
    if not has_hallucination:
        continue

    ann_ids = coco_caps.getAnnIds(imgIds=img_id)
    if not ann_ids:
        continue
    gt_captions = [a['caption'].strip() for a in coco_caps.loadAnns(ann_ids)]

    hal_caption = cache_info['caption'].strip()
    for gt_caption in gt_captions:  # up to 5 GT captions per image
        dpo_pairs.append({
            'img_id':   img_id,
            'img_path': img_path,
            'chosen':   gt_caption,   # preferred: grounded GT caption
            'rejected': hal_caption,  # dispreferred: model's own hallucination
        })

random.shuffle(dpo_pairs)
print(f'\nDPO training pairs : {len(dpo_pairs)}')
print(f'Unique train images : {len(set(p["img_id"] for p in dpo_pairs))}')
print(f'Eval images         : {len(eval_images)}')
print(f'\nExample pair:')
print(f'  chosen  : {dpo_pairs[0]["chosen"]}')
print(f'  rejected: {dpo_pairs[0]["rejected"]}')

loading annotations into memory...
Done (t=1.13s)
creating index...
index created!
COCO captions loaded: 202654 annotations

DPO training pairs : 321
Unique train images : 64
Eval images         : 40

Example pair:
  chosen  : Man posing in front of bicycle with a banana in his hand.
  rejected: The image features a man wearing sunglasses and a yellow shirt, posing with a yellow bicycle. He is standing next to the bicycle, which is prominently displayed in the scene. The man appears to be proud of his bicycle, possibly showcasing it to others.

In the background, there is a chair and a door,


## 6. Define evaluation helper functions

Self-contained: COCO vocabulary, caption generation, POPE, and CHAIR — no imports from Stage 1.

In [6]:
import spacy
nlp = spacy.load('en_core_web_sm')

# COCO synonym map and single-word aliases (mirrors Stage 1)
COCO_SYNONYMS = {
    'person':        ['man','woman','people','boy','girl','child','guy','lady','kid',
                      'baby','player','rider','skier','surfer','snowboarder'],
    'car':           ['vehicle','automobile','sedan','suv'],
    'dog':           ['puppy','dogs'],
    'cat':           ['kitten','cats'],
    'tv':            ['television','monitor','screen'],
    'couch':         ['sofa'],
    'cell phone':    ['phone','cellphone','smartphone'],
    'dining table':  ['table','desk'],
    'wine glass':    ['glass'],
    'bicycle':       ['bike'],
    'motorcycle':    ['motorbike'],
    'airplane':      ['plane','jet'],
    'potted plant':  ['plant'],
    'laptop':        ['computer'],
    'refrigerator':  ['fridge'],
    'truck':         ['lorry'],
    'boat':          ['ship','sailboat'],
    'fire hydrant':  ['hydrant'],
    'hot dog':       ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball':   ['ball','football','soccer ball','basketball'],
    'baseball bat':  ['bat'],
    'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant':'fire hydrant','hotdog':'hot dog','stoplight':'traffic light',
    'bat':'baseball bat','racket':'tennis racket','racquet':'tennis racket',
}
OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())


@torch.no_grad()
def generate_caption(model_obj, image_path, max_new_tokens=80):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()
    out = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             return_dict_in_generate=True, use_cache=True)
    gen_ids = out.sequences[0, inputs['input_ids'].shape[1]:]
    caption = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    return caption, gen_ids.cpu()


def find_content_words(gen_ids, gt_objects):
    """Identify nouns in generated caption; label as object/hallucinated. Mirror of Stage 1."""
    full_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    gt_norm = set(o.lower() for o in gt_objects)
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)
    for alias, canonical in MULTIWORD_ALIASES.items():
        if canonical in gt_norm:
            expanded_gt.add(alias)

    doc = nlp(full_text)
    nouns = [tok.text.lower().strip() for tok in doc
             if tok.pos_ in ('NOUN', 'PROPN', 'ADJ') and len(tok.text.strip()) >= 2]
    if not nouns:
        return []

    accumulated = ''
    results = []
    noun_idx = 0
    for tok_i, tid in enumerate(gen_ids):
        if noun_idx >= len(nouns):
            break
        ts = processor.tokenizer.decode([int(tid)], skip_special_tokens=True).lower()
        accumulated += ts
        target = nouns[noun_idx]
        if target in accumulated:
            canonical = MULTIWORD_ALIASES.get(target, target)
            is_object = target in OBJECT_VOCAB
            is_hall   = is_object and (target not in expanded_gt) and (canonical not in expanded_gt)
            results.append({'word': target, 'is_object': is_object, 'is_hallucinated': is_hall})
            cut = accumulated.rfind(target) + len(target)
            accumulated = accumulated[cut:]
            noun_idx += 1
    return results


@torch.no_grad()
def pope_eval(model_obj, images, gt_objects_list, n_per_image=6, seed=0):
    """POPE-style binary evaluation: 'Is there a {obj} in the image?' → yes/no."""
    rng = random.Random(seed)
    coco_cats = list(ALL_COCO_OBJECTS)
    preds, labels = [], []

    model_obj.eval()
    for img_id, gt_set in tqdm(zip(images, gt_objects_list), total=len(images), desc='POPE'):
        img_path = img_id_to_path.get(img_id)
        if not img_path or not os.path.exists(img_path):
            continue
        img = Image.open(img_path).convert('RGB')

        pos_objects = rng.sample(list(gt_set), min(n_per_image // 2, len(gt_set)))
        neg_pool    = [c for c in coco_cats if c not in gt_set]
        neg_objects = rng.sample(neg_pool, min(n_per_image // 2, len(neg_pool)))

        for obj, gt_label in [(o, 1) for o in pos_objects] + [(o, 0) for o in neg_objects]:
            q = f"Is there a {obj} in the image? Please answer yes or no."
            prompt = f"USER: <image>\n{q}\nASSISTANT:"
            inputs = processor(text=prompt, images=img, return_tensors='pt').to(device, torch.float16)
            inputs['input_ids'] = inputs['input_ids'].long()
            inputs['attention_mask'] = inputs['attention_mask'].long()
            out = model_obj.generate(**inputs, max_new_tokens=5, do_sample=False)
            answer = processor.tokenizer.decode(
                out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).lower()
            preds.append(1 if 'yes' in answer else 0)
            labels.append(gt_label)

        torch.cuda.empty_cache()

    preds  = np.array(preds)
    labels = np.array(labels)
    acc    = (preds == labels).mean()
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    return {'accuracy': float(acc), 'precision': float(prec),
            'recall': float(rec), 'f1': float(f1),
            'yes_rate': float(preds.mean()), 'n': len(preds)}


@torch.no_grad()
def chair_eval(model_obj, images, gt_objects_list):
    """CHAIR-style evaluation on generated captions."""
    model_obj.eval()
    chairs_list, chairi_list = [], []

    for img_id, gt_set in tqdm(zip(images, gt_objects_list), total=len(images), desc='CHAIR'):
        img_path = img_id_to_path.get(img_id)
        if not img_path or not os.path.exists(img_path):
            continue
        caption, gen_ids = generate_caption(model_obj, img_path)
        cw = find_content_words(gen_ids, gt_set)
        obj_words  = [c for c in cw if c['is_object']]
        hall_words = [c for c in cw if c['is_hallucinated']]
        chairs_list.append(1 if hall_words else 0)
        chairi_list.append(len(hall_words) / max(len(obj_words), 1))
        torch.cuda.empty_cache()

    return {'CHAIRs': float(np.mean(chairs_list)),
            'CHAIRi': float(np.mean(chairi_list)),
            'n': len(chairs_list)}


print('Eval functions defined.')

Eval functions defined.


## 7. Baseline evaluation (base model, before LoRA)

Run POPE and CHAIR on the held-out 40 eval images. Takes ~15 minutes.

In [7]:
print('=== BASELINE EVALUATION (no LoRA) ===')
baseline_pope  = pope_eval(model, eval_images, eval_gt_objects)
baseline_chair = chair_eval(model, eval_images, eval_gt_objects)

baseline_results = {'pope': baseline_pope, 'chair': baseline_chair}
with open(f'{WORK_DIR}/results/stage2_baseline_eval.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print('\nPOPE  :', baseline_pope)
print('CHAIR :', baseline_chair)

=== BASELINE EVALUATION (no LoRA) ===


POPE:   0%|          | 0/40 [00:00<?, ?it/s]

CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]


POPE  : {'accuracy': 0.9083333333333333, 'precision': 0.99, 'recall': 0.825, 'f1': 0.9, 'yes_rate': 0.4166666666666667, 'n': 240}
CHAIR : {'CHAIRs': 0.35, 'CHAIRi': 0.10965277777777778, 'n': 40}


## 8. Apply surgical LoRA (PEFT)

`layers_to_transform` restricts LoRA to the 17 layers that contain at least one hallucination head.  
Gradient masking in the next cell further restricts updates to only the 32 identified head-slices.

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj'],
    layers_to_transform=target_layers,  # only layers with identified hallucination heads
    lora_dropout=0.05,
    bias='none',
)

model_lora = get_peft_model(model, lora_config)

# Disable any LoRA accidentally added to the vision tower
# (CLIP's self-attn also has q/k/v_proj; layer indices overlap with target_layers)
n_vision_disabled = 0
for name, param in model_lora.named_parameters():
    if ('vision_tower' in name or 'multi_modal_projector' in name) and 'lora_' in name:
        param.requires_grad = False
        n_vision_disabled += 1

model_lora.print_trainable_parameters()
print(f'Vision tower LoRA params disabled: {n_vision_disabled}')

trainable params: 3,735,552 || all params: 7,067,752,448 || trainable%: 0.0529
Vision tower LoRA params disabled: 72


## 9. Register gradient masking hooks

Each `lora_B.weight` has shape `[4096, r]`. Head `h` occupies rows `[h*128 : (h+1)*128]`.
The hook zeros out gradient rows for heads NOT in the hallucination shortlist, so only the 32 target head-slices are updated.

In [9]:
grad_hooks = []
hooks_registered = 0

for name, param in model_lora.named_parameters():
    if 'lora_B' not in name or not param.requires_grad:
        continue
    if 'vision_tower' in name or 'multi_modal_projector' in name:
        continue

    m = re.search(r'layers\.([0-9]+)\.self_attn', name)
    if m is None:
        continue
    layer_idx = int(m.group(1))
    if layer_idx not in hal_heads_by_layer:
        continue

    # Build binary mask: 1 for target head rows, 0 for all others
    # param.shape = [out_features, r] = [4096, 8]
    out_size = param.shape[0]   # 4096 = 32 heads * 128 head_dim
    mask = torch.zeros(out_size, device=device)
    for h in hal_heads_by_layer[layer_idx]:
        mask[h * HEAD_DIM : (h + 1) * HEAD_DIM] = 1.0

    # Closure captures the correct mask for this parameter
    # grad shape: [4096, r]; mask.unsqueeze(-1): [4096, 1] → broadcasts over rank dim
    handle = param.register_hook(lambda g, m=mask: g * m.unsqueeze(-1))
    grad_hooks.append(handle)
    hooks_registered += 1

    active_heads = len(hal_heads_by_layer[layer_idx])
    proj_name = name.split('.')[-4] if '.' in name else '?'
    print(f'  L{layer_idx:02d} {proj_name}: {active_heads}/{NUM_HEADS} heads active')

print(f'\nTotal grad-masking hooks: {hooks_registered}')
print(f'Each hook restricts lora_B updates to {len(final_list)} head-slices total')

  L07 q_proj: 1/32 heads active
  L07 k_proj: 1/32 heads active
  L07 v_proj: 1/32 heads active
  L09 q_proj: 1/32 heads active
  L09 k_proj: 1/32 heads active
  L09 v_proj: 1/32 heads active
  L12 q_proj: 3/32 heads active
  L12 k_proj: 3/32 heads active
  L12 v_proj: 3/32 heads active
  L13 q_proj: 1/32 heads active
  L13 k_proj: 1/32 heads active
  L13 v_proj: 1/32 heads active
  L14 q_proj: 1/32 heads active
  L14 k_proj: 1/32 heads active
  L14 v_proj: 1/32 heads active
  L15 q_proj: 1/32 heads active
  L15 k_proj: 1/32 heads active
  L15 v_proj: 1/32 heads active
  L16 q_proj: 1/32 heads active
  L16 k_proj: 1/32 heads active
  L16 v_proj: 1/32 heads active
  L17 q_proj: 2/32 heads active
  L17 k_proj: 2/32 heads active
  L17 v_proj: 2/32 heads active
  L19 q_proj: 1/32 heads active
  L19 k_proj: 1/32 heads active
  L19 v_proj: 1/32 heads active
  L21 q_proj: 1/32 heads active
  L21 k_proj: 1/32 heads active
  L21 v_proj: 1/32 heads active
  L22 q_proj: 2/32 heads active
  L22 k_

In [10]:
# --- Verify masking: non-target rows should have zero gradient after backward ---
print('Verifying gradient masking ...')
model_lora.train()

_test_img  = Image.open(dpo_pairs[0]['img_path']).convert('RGB')
_test_text = PROMPT_TEMPLATE + dpo_pairs[0]['chosen']
_inputs    = processor(text=_test_text, images=_test_img, return_tensors='pt').to(device, torch.float16)
_inputs['input_ids'] = _inputs['input_ids'].long()
_inputs['attention_mask'] = _inputs['attention_mask'].long()

_out  = model_lora(**_inputs, return_dict=True)
_loss = _out.logits.float().mean()   # dummy loss
_loss.backward()

check_ok = True
for name, param in model_lora.named_parameters():
    if 'lora_B' not in name or param.grad is None:
        continue
    if 'vision_tower' in name:
        continue
    m = re.search(r'layers\.([0-9]+)\.self_attn', name)
    if m is None:
        continue
    layer_idx = int(m.group(1))
    if layer_idx not in hal_heads_by_layer:
        continue

    grad = param.grad  # [4096, r]
    for h in range(NUM_HEADS):
        row_slice = grad[h * HEAD_DIM : (h + 1) * HEAD_DIM]
        norm = row_slice.norm().item()
        is_target = h in hal_heads_by_layer[layer_idx]
        if not is_target and norm > 1e-6:
            print(f'  FAIL: L{layer_idx} head {h} (non-target) grad norm = {norm:.6f}')
            check_ok = False

if check_ok:
    print('PASS: All non-target head rows have zero gradient.')

# Clean up verification pass
model_lora.zero_grad()
del _out, _loss, _inputs, _test_img
torch.cuda.empty_cache()

Verifying gradient masking ...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


PASS: All non-target head rows have zero gradient.


## 10. Compute PROMPT_LEN

`PROMPT_LEN` is the number of input tokens for the bare prompt (no caption). It is constant across all images because the prompt template is fixed and CLIP always produces 576 patch tokens. We use it to mask the prompt from the DPO loss (labels = -100 for prompt tokens).

In [11]:
_dummy_img    = Image.open(dpo_pairs[0]['img_path']).convert('RGB')
_prompt_in    = processor(text=PROMPT_TEMPLATE, images=_dummy_img, return_tensors='pt')
_raw_len      = _prompt_in['input_ids'].shape[1]
_n_img_ph     = (_prompt_in['input_ids'] == IMAGE_TOKEN_ID).sum().item()

# New transformers (>=4.47): processor pre-expands to 576; single-placeholder = old behaviour
if _n_img_ph >= NUM_IMAGE_TOKENS:
    PROMPT_LEN = _raw_len
else:
    PROMPT_LEN = _raw_len - 1 + NUM_IMAGE_TOKENS

del _dummy_img, _prompt_in
print(f'PROMPT_LEN = {PROMPT_LEN} (raw input_ids len = {_raw_len}, '
      f'img placeholders = {_n_img_ph})')

PROMPT_LEN = 595 (raw input_ids len = 595, img placeholders = 576)


## 11. DPO Training Loop

**Loss:**
$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left( \beta (\log\pi_\theta(y_w|x) - \log\pi_{\text{ref}}(y_w|x)) - \beta (\log\pi_\theta(y_l|x) - \log\pi_{\text{ref}}(y_l|x)) \right)$$

- $y_w$ = GT caption (chosen), $y_l$ = hallucinated caption (rejected)
- $\pi_{\text{ref}}$ = base model computed via `disable_adapter_layers()` (no extra VRAM)
- Sequence logprob = mean per-token log-prob over caption tokens only

In [12]:
def make_labels(input_ids, prompt_len):
    """Return labels with -100 for prompt tokens; caption token ids for the rest."""
    labels = torch.full_like(input_ids, -100)
    labels[:, prompt_len:] = input_ids[:, prompt_len:]
    return labels


def sequence_logprob(logits, labels):
    """
    Mean per-token log-prob over labeled positions (next-token prediction).
    logits: [1, T, V] (fp16 or fp32)
    labels: [1, T] with -100 for ignored positions
    Returns: scalar tensor
    """
    shift_logits = logits[:, :-1, :].float()   # [1, T-1, V]
    shift_labels = labels[:, 1:]               # [1, T-1]

    log_probs = F.log_softmax(shift_logits, dim=-1)  # [1, T-1, V]
    mask      = shift_labels != -100
    safe_lbl  = shift_labels.clone()
    safe_lbl[~mask] = 0

    token_lp  = log_probs.gather(2, safe_lbl.unsqueeze(-1)).squeeze(-1)  # [1, T-1]
    token_lp  = token_lp * mask.float()
    return token_lp.sum() / mask.float().sum().clamp_min(1)              # scalar


# --- Hyperparameters ---
BETA          = 0.1
LR            = 1e-4
NUM_EPOCHS    = 1
GRAD_ACCUM    = 4
MAX_GRAD_NORM = 1.0

total_grad_steps = (len(dpo_pairs) * NUM_EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM

optimizer = torch.optim.AdamW(
    [p for p in model_lora.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_grad_steps, eta_min=LR / 10
)

# Checkpoint/resume
STAGE2_CKPT = f'{WORK_DIR}/cache/stage2_checkpoint.json'
start_epoch  = 0
training_log = []
if os.path.exists(STAGE2_CKPT):
    with open(STAGE2_CKPT) as f:
        ckpt = json.load(f)
    start_epoch  = ckpt.get('completed_epochs', 0)
    training_log = ckpt.get('log', [])
    print(f'Resuming from epoch {start_epoch}')

print(f'Training: {len(dpo_pairs)} pairs × {NUM_EPOCHS} epochs')
print(f'Grad updates: {total_grad_steps}  (accum={GRAD_ACCUM})')

Training: 321 pairs × 3 epochs
Grad updates: 241  (accum=4)


In [13]:
model_lora.train()
optimizer.zero_grad()
global_step = 0

for epoch in range(start_epoch, NUM_EPOCHS):
    random.shuffle(dpo_pairs)
    epoch_loss = 0.0
    n_steps    = 0
    pbar = tqdm(enumerate(dpo_pairs), total=len(dpo_pairs),
                desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}')

    for step, pair in pbar:
        try:
            img = Image.open(pair['img_path']).convert('RGB')

            # --- Chosen (GT caption) ---
            chosen_in = processor(
                text=PROMPT_TEMPLATE + pair['chosen'], images=img,
                return_tensors='pt').to(device, torch.float16)
            chosen_in['input_ids']      = chosen_in['input_ids'].long()
            chosen_in['attention_mask'] = chosen_in['attention_mask'].long()
            chosen_labels = make_labels(chosen_in['input_ids'], PROMPT_LEN).to(device)

            # --- Rejected (hallucinated caption) ---
            rejected_in = processor(
                text=PROMPT_TEMPLATE + pair['rejected'], images=img,
                return_tensors='pt').to(device, torch.float16)
            rejected_in['input_ids']      = rejected_in['input_ids'].long()
            rejected_in['attention_mask'] = rejected_in['attention_mask'].long()
            rejected_labels = make_labels(rejected_in['input_ids'], PROMPT_LEN).to(device)

            # --- Reference log-probs (LoRA disabled, no gradient) ---
            model_lora.disable_adapter_layers()
            with torch.no_grad():
                ref_lp_chosen   = sequence_logprob(
                    model_lora(**chosen_in,   return_dict=True).logits, chosen_labels)
                ref_lp_rejected = sequence_logprob(
                    model_lora(**rejected_in, return_dict=True).logits, rejected_labels)
            model_lora.enable_adapter_layers()

            # --- Policy log-probs (LoRA enabled, with gradient) ---
            lp_chosen   = sequence_logprob(
                model_lora(**chosen_in,   return_dict=True).logits, chosen_labels)
            lp_rejected = sequence_logprob(
                model_lora(**rejected_in, return_dict=True).logits, rejected_labels)

            # --- DPO loss ---
            delta_chosen   = BETA * (lp_chosen   - ref_lp_chosen.detach())
            delta_rejected = BETA * (lp_rejected - ref_lp_rejected.detach())
            loss = -F.logsigmoid(delta_chosen - delta_rejected)

            (loss / GRAD_ACCUM).backward()
            epoch_loss += loss.item()
            n_steps    += 1

            if (step + 1) % GRAD_ACCUM == 0 or step == len(dpo_pairs) - 1:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model_lora.parameters() if p.requires_grad],
                    MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 20 == 0:
                    pbar.set_postfix({
                        'loss': f'{epoch_loss / n_steps:.4f}',
                        'lr':   f'{scheduler.get_last_lr()[0]:.2e}',
                    })

            del chosen_in, rejected_in, chosen_labels, rejected_labels
            torch.cuda.empty_cache()

        except Exception as e:
            print(f'\nStep {step} failed: {type(e).__name__}: {e}')
            optimizer.zero_grad()
            torch.cuda.empty_cache()

    avg_loss = epoch_loss / max(n_steps, 1)
    training_log.append({'epoch': epoch + 1, 'loss': avg_loss, 'steps': n_steps})
    print(f'Epoch {epoch + 1}: avg_loss={avg_loss:.4f}  steps={n_steps}')

    with open(STAGE2_CKPT, 'w') as f:
        json.dump({'completed_epochs': epoch + 1, 'log': training_log}, f, indent=2)

print('\nTraining complete.')
print('Training log:', training_log)

Epoch 1/3:   0%|          | 0/321 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch 1: avg_loss=0.6677  steps=321


Epoch 2/3:   0%|          | 0/321 [00:00<?, ?it/s]

Epoch 2: avg_loss=0.5801  steps=321


Epoch 3/3:   0%|          | 0/321 [00:00<?, ?it/s]

Epoch 3: avg_loss=0.3344  steps=321

Training complete.
Training log: [{'epoch': 1, 'loss': 0.6676941616141536, 'steps': 321}, {'epoch': 2, 'loss': 0.5800813793021942, 'steps': 321}, {'epoch': 3, 'loss': 0.33435312038827164, 'steps': 321}]


## 12. Post-training evaluation (LoRA enabled)

In [14]:
model_lora.enable_adapter_layers()

print('=== POST-TRAINING EVALUATION (with surgical LoRA) ===')
# lora_pope  = pope_eval(model_lora, eval_images, eval_gt_objects)
lora_chair = chair_eval(model_lora, eval_images, eval_gt_objects)

lora_results = {'pope': lora_pope, 'chair': lora_chair}
with open(f'{WORK_DIR}/results/stage2_lora_eval.json', 'w') as f:
    json.dump(lora_results, f, indent=2)

# print('\nPOPE  :', lora_pope)
print('CHAIR :', lora_chair)

=== POST-TRAINING EVALUATION (with surgical LoRA) ===


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

NameError: name 'lora_pope' is not defined

In [15]:
lora_results = {'chair': lora_chair}
with open(f'{WORK_DIR}/results/stage2_lora_eval.json', 'w') as f:
    json.dump(lora_results, f, indent=2)

# print('\nPOPE  :', lora_pope)
print('CHAIR :', lora_chair)

CHAIR : {'CHAIRs': 0.0, 'CHAIRi': 0.0, 'n': 40}


## 13. Comparison table

In [16]:
b = baseline_results
l = lora_results

def d(after, before, higher_is_better=True):
    delta = after - before
    sign  = '+' if delta >= 0 else ''
    arrow = '↑' if (delta > 0) == higher_is_better else ('↓' if delta != 0 else '=')
    return f'{sign}{delta:.4f} {arrow}'

rows = [
    # ('POPE Accuracy',  b['pope']['accuracy'],  l['pope']['accuracy'],  True),
    # ('POPE F1',        b['pope']['f1'],         l['pope']['f1'],         True),
    # ('POPE Precision', b['pope']['precision'],  l['pope']['precision'],  True),
    # ('POPE Recall',    b['pope']['recall'],     l['pope']['recall'],     True),
    # ('POPE Yes-rate',  b['pope']['yes_rate'],   l['pope']['yes_rate'],   None),
    ('CHAIRs',         b['chair']['CHAIRs'],    l['chair']['CHAIRs'],    False),
    ('CHAIRi',         b['chair']['CHAIRi'],    l['chair']['CHAIRi'],    False),
]

print('=' * 62)
print(f'{"Metric":<20} {"Baseline":>10} {"LoRA":>10} {"Delta":>16}')
print('-' * 62)
for name, bv, lv, hib in rows:
    delta_str = d(lv, bv, hib) if hib is not None else f'{lv - bv:+.4f}'
    print(f'{name:<20} {bv:>10.4f} {lv:>10.4f} {delta_str:>16}')
print('=' * 62)
print(f'Training pairs used : {len(dpo_pairs)}')
print(f'Epochs              : {NUM_EPOCHS}')
print(f'Target heads (LoRA) : {len(final_list)} / 1024 ({100*len(final_list)/1024:.1f}%)')

Metric                 Baseline       LoRA            Delta
--------------------------------------------------------------
CHAIRs                   0.3500     0.0000        -0.3500 ↑
CHAIRi                   0.1097     0.0000        -0.1097 ↑
Training pairs used : 321
Epochs              : 3
Target heads (LoRA) : 32 / 1024 (3.1%)


## 14. Save LoRA adapter

In [17]:
adapter_dir = f'{WORK_DIR}/results/stage2_lora_adapter'
model_lora.save_pretrained(adapter_dir)
print(f'LoRA adapter saved to: {adapter_dir}')
print('Files:')
for f_name in os.listdir(adapter_dir):
    fpath = os.path.join(adapter_dir, f_name)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {f_name:<40} {size_mb:.1f} MB')

# Save training log
log_path = f'{WORK_DIR}/results/stage2_training_log.json'
with open(log_path, 'w') as f:
    json.dump({'hyperparams': {'BETA': BETA, 'LR': LR, 'NUM_EPOCHS': NUM_EPOCHS,
                               'GRAD_ACCUM': GRAD_ACCUM, 'lora_rank': 8, 'lora_alpha': 16,
                               'n_target_heads': len(final_list), 'n_target_layers': len(target_layers)},
               'training_log': training_log,
               'baseline': baseline_results,
               'lora_eval': lora_results}, f, indent=2)
print(f'Training log saved to: {log_path}')

LoRA adapter saved to: /content/drive/MyDrive/llava_hallucination_heads/results/stage2_lora_adapter
Files:
  README.md                                0.0 MB
  adapter_model.safetensors                17.3 MB
  adapter_config.json                      0.0 MB
Training log saved to: /content/drive/MyDrive/llava_hallucination_heads/results/stage2_training_log.json


## Stage 2 Complete

### What was done
- Loaded LLaVA-1.5-7B in **4-bit (QLoRA)** — safe because Stage 2 does not extract attention weights
- Applied **LoRA (r=8, α=16)** exclusively to Q/K/V projections of the **17 layers** that contain identified hallucination heads
- Registered **gradient masking hooks** on `lora_B` weights so only the **32 target head-slices** (out of 1,024) are updated — all other rows of the LoRA adapter receive zero gradient
- Trained with **DPO loss** on ~500–800 contrastive COCO val2014 pairs:  
  - Chosen: COCO ground-truth captions  
  - Rejected: model's own hallucinated captions from Stage 1  
- Reference logprobs computed inline via `disable_adapter_layers()` — no duplicate model in VRAM

### Outputs saved to Drive
| File | Description |
|------|-------------|
| `results/stage2_lora_adapter/` | LoRA adapter weights (~10–30 MB) |
| `results/stage2_baseline_eval.json` | POPE + CHAIR before LoRA |
| `results/stage2_lora_eval.json` | POPE + CHAIR after LoRA |
| `results/stage2_training_log.json` | Epoch losses + hyperparams |

### Next: Stage 3 — Inference-Time Grounding Score
Monitor the 32 identified heads' visual attention ratio at each generation step.  
When the ratio drops below a threshold, apply logit penalties to content-word tokens.  
This is training-free and stacks on top of the Stage 2 LoRA adapter.